# Cosmos DB Live Container Migration - Parameterized (Fabric)

This notebook performs a **live migration** of a single Cosmos DB container using parameters passed from a parent notebook. It is designed to be called by `04_CosmosDB_Parallel_Container_Migration` for multi-container scenarios.

Converted from [CosmosDBLiveContainerMigration.scala](https://github.com/Azure/azure-sdk-for-java/tree/main/sdk/cosmos/azure-cosmos-spark_3/Samples/DatabricksLiveContainerMigration).

**How it works:**
| Step | Description |
|------|-------------|
| 1 | Receive parameters (endpoint, keys, database, container, etc.) |
| 2 | Configure Spark Cosmos DB catalog |
| 3 | Create target database, throughput control table, and target container |
| 4 | Initialize change feed reader and writer |
| 5 | Stream migration with count-based completion verification |

**Completion strategy:** The stream runs continuously while periodic count queries on both containers determine when migration is complete. This is immune to throttling, duplicate change feed events, and checkpoint state — only actual document counts decide when to stop.

> **Note:** This notebook is called via `notebookutils.notebook.run()` or `runMultiple()`. Parameters are injected into the parameter cell below.

## Step 1: Configure Spark Session with Cosmos DB Spark Connector

In [1]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "Cosmos_Migration"
    }
}

StatementMeta(, d44d3ec3-4eba-4ca1-85fa-40aa2de2be4a, -1, Finished, Available, Finished, True)

In [2]:
# JAR Loading: SKIPPED - Using Fabric Environment
# The Fabric Environment "CosmosDB_Migration_Env" already provides:
#   /usr/lib/library-manager/bin/libraries/scala/azure-cosmos-spark_3-5_2-12-4.41.0.jar
# Attempting to add it again causes "already registered" errors.
print("✅ Using Cosmos Spark JAR from Fabric Environment")
print("   Path: /usr/lib/library-manager/bin/libraries/scala/azure-cosmos-spark_3-5_2-12-4.41.0.jar")

StatementMeta(, d44d3ec3-4eba-4ca1-85fa-40aa2de2be4a, 5, Finished, Available, Finished, False)

✅ Using Cosmos Spark JAR from Fabric Environment
   Path: /usr/lib/library-manager/bin/libraries/scala/azure-cosmos-spark_3-5_2-12-4.41.0.jar


## Step 2: Parameters (Injected by Parent Notebook)

These default values are overridden when this notebook is called via `notebookutils.notebook.run()` with arguments.

> **Important:** Mark this cell as a **parameter cell** in the Fabric notebook UI so arguments from the parent notebook are injected correctly.

In [ ]:
# =============================================================================
# PARAMETER CELL — Mark this cell as "parameters" in the notebook UI
# These defaults are overridden when called from 04_CosmosDB_Parallel_Container_Migration
# =============================================================================

# Source config
cosmos_source_endpoint = "https://sourcedata.documents.azure.com:443/"  # Cosmos DB Account URI of the source account
cosmos_source_masterkey = ""  # Cosmos DB Account PRIMARY KEY of the source account
cosmos_source_database_name = "SampleDB"  # name of your source database
cosmos_source_container_name = "SampleContainer"  # name of the container you want to migrate
cosmos_source_throughput_control = "0.95"  # target percentage of available throughput for migration

# Target config
cosmos_target_endpoint = "https://targetdata.documents.azure.com:443/"  # Cosmos DB Account URI of the target account
cosmos_target_masterkey = ""  # Cosmos DB Account PRIMARY KEY of the target account
cosmos_target_database_name = "SampleDB"  # name of your target database
cosmos_target_container_name = "SampleContainer5"  # name of the target container
cosmos_target_container_partition_key = "/categoryId"  # partition key field for the target container (keep forward slash)
cosmos_target_container_provisioned_throughput = "10000"  # provisioned throughput for the target container

# Common config
cosmos_region = "[East US]"
clear_checkpoint = "false"  # set to "true" only for a full re-migration from scratch

## Step 3: Configure Cosmos DB Spark Catalog

In [ ]:
# Configure SOURCE Cosmos DB Spark Catalog
spark.conf.set("spark.sql.catalog.cosmosCatalogSource", "com.azure.cosmos.spark.CosmosCatalog")
spark.conf.set("spark.sql.catalog.cosmosCatalogSource.spark.cosmos.accountEndpoint", cosmos_source_endpoint)
spark.conf.set("spark.sql.catalog.cosmosCatalogSource.spark.cosmos.accountKey", cosmos_source_masterkey)

# Configure TARGET Cosmos DB Spark Catalog
spark.conf.set("spark.sql.catalog.cosmosCatalogTarget", "com.azure.cosmos.spark.CosmosCatalog")
spark.conf.set("spark.sql.catalog.cosmosCatalogTarget.spark.cosmos.accountEndpoint", cosmos_target_endpoint)
spark.conf.set("spark.sql.catalog.cosmosCatalogTarget.spark.cosmos.accountKey", cosmos_target_masterkey)

print(f"SOURCE Cosmos DB: {cosmos_source_endpoint}")
print(f"  Database/Container: {cosmos_source_database_name}/{cosmos_source_container_name}")
print(f"TARGET Cosmos DB: {cosmos_target_endpoint}")
print(f"  Database/Container: {cosmos_target_database_name}/{cosmos_target_container_name}")

In [ ]:
# SKIP driver-side verification - doesn't work in Fabric child notebooks
# The JAR is loaded on executors where Cosmos operations actually run
# If Cosmos operations fail, we'll get better error messages than this check provides
print("⏭️ Skipping JAR verification (Fabric child notebooks limitation)")
print("   JAR was added to Spark executors - proceeding with migration")
print("   If Cosmos operations fail, check that JAR downloaded successfully")

## Step 4: Create Target Database, Throughput Control & Target Container

- Creates the **target database** if it doesn't exist (handles shared throughput databases)
- Creates a **ThroughputControl** table in the source database for rate-limiting
- Creates the **target container** with specified partition key and throughput

In [ ]:
def get_database_properties(provisioned_throughput):
    """Returns DBPROPERTIES clause for shared throughput databases."""
    if "shared" in provisioned_throughput.lower():
        throughput_value = provisioned_throughput.replace("sharedDB", "").strip()
        return f"WITH DBPROPERTIES (autoScaleMaxThroughput = '{throughput_value}')"
    return ""

def get_container_throughput_clause(provisioned_throughput):
    """Returns throughput clause for dedicated container throughput."""
    if "shared" not in provisioned_throughput.lower():
        return f"autoScaleMaxThroughput = '{provisioned_throughput}',"
    return ""

# --- Create Target Database (IF NOT EXISTS) ---
db_properties = get_database_properties(cosmos_target_container_provisioned_throughput)
create_db_query = f"CREATE DATABASE IF NOT EXISTS cosmosCatalogTarget.`{cosmos_target_database_name}` {db_properties};"
print(f"✓ Creating target database: {cosmos_target_database_name}")
try:
    spark.sql(create_db_query)
    print(f"  ✅ Database '{cosmos_target_database_name}' created/verified in target")
except Exception as e:
    print(f"  ⚠️ Database creation issue: {str(e)[:100]}")
    # Continue anyway - database might already exist

# --- Create Throughput Control Table (IF NOT EXISTS) ---
create_tc_query = f"""
CREATE TABLE IF NOT EXISTS cosmosCatalogSource.`{cosmos_source_database_name}`.`ThroughputControl`
USING cosmos.oltp
OPTIONS (spark.cosmos.database = '{cosmos_source_database_name}')
TBLPROPERTIES(
    partitionKeyPath = '/groupId',
    autoScaleMaxThroughput = '4000',
    indexingPolicy = 'AllProperties',
    defaultTtlInSeconds = '-1'
)
"""
print(f"✓ Creating throughput control table in source database...")
try:
    spark.sql(create_tc_query)
    print(f"  ✅ ThroughputControl table created/verified in source")
except Exception as e:
    print(f"  ⚠️ ThroughputControl creation issue: {str(e)[:100]}")
    # Continue anyway - table might already exist

# --- Create Target Container (IF NOT EXISTS) ---
container_throughput = get_container_throughput_clause(cosmos_target_container_provisioned_throughput)
create_container_query = f"""
CREATE TABLE IF NOT EXISTS cosmosCatalogTarget.`{cosmos_target_database_name}`.`{cosmos_target_container_name}`
USING cosmos.oltp
TBLPROPERTIES(
    partitionKeyPath = '{cosmos_target_container_partition_key}',
    {container_throughput}
    indexingPolicy = 'OnlySystemProperties'
)
"""
print(f"✓ Creating target container: {cosmos_target_database_name}/{cosmos_target_container_name}")
print(f"  Partition key: {cosmos_target_container_partition_key}")
try:
    spark.sql(create_container_query)
    print(f"  ✅ Container '{cosmos_target_container_name}' created/verified in target")
except Exception as e:
    print(f"  ❌ Container creation FAILED: {str(e)[:200]}")
    raise Exception(f"Cannot proceed - target container creation failed: {e}")

## Step 5: Initialize Change Feed Reader & Writer

Configures the change feed reader (source) and writer (target). Checkpoint location is stored in the Lakehouse Files area per source/target pair, enabling resume after failures.

> **Completion strategy:** The stream runs continuously while periodic count queries on source and target containers determine when migration is complete. This is immune to throttling, duplicate change feed events, and checkpoint state — only actual document counts matter.

In [ ]:
# Change feed reader configuration (SOURCE)
# Ensure cosmos_region is defined (fallback if parameter cell didn't set it)
if 'cosmos_region' not in dir():
    cosmos_region = "[East US]"

change_feed_cfg = {
    "spark.cosmos.accountEndpoint": cosmos_source_endpoint,
    "spark.cosmos.accountKey": cosmos_source_masterkey,
    "spark.cosmos.applicationName": f"{cosmos_source_database_name}_{cosmos_source_container_name}_LiveMigrationRead_",
    "spark.cosmos.database": cosmos_source_database_name,
    "spark.cosmos.container": cosmos_source_container_name,
    "spark.cosmos.read.partitioning.strategy": "Restrictive",
    "spark.cosmos.read.inferSchema.enabled": "false",
    "spark.cosmos.read.maxItemCount": "5",
    "spark.cosmos.changeFeed.startFrom": "Beginning",
    "spark.cosmos.changeFeed.mode": "Incremental",
    "spark.cosmos.changeFeed.itemCountPerTriggerHint": "50000",
    # Throughput control
    "spark.cosmos.throughputControl.enabled": "true",
    "spark.cosmos.throughputControl.name": "SourceContainerThroughputControl",
    "spark.cosmos.throughputControl.targetThroughputThreshold": cosmos_source_throughput_control,
    "spark.cosmos.throughputControl.globalControl.database": cosmos_source_database_name,
    "spark.cosmos.throughputControl.globalControl.container": "ThroughputControl",
    "spark.cosmos.preferredRegionsList": cosmos_region,
}

# Checkpoint location (per source+target db/container) in Lakehouse Files area
checkpoint_location = f"Files/cosmos_migration_checkpoints/{cosmos_source_database_name}/{cosmos_source_container_name}_to_{cosmos_target_database_name}/{cosmos_target_container_name}"
application_name = f"{cosmos_source_database_name}_{cosmos_source_container_name}_to_{cosmos_target_container_name}_"

# Clear stale checkpoint if requested
if clear_checkpoint.lower() == "true":
    import subprocess
    checkpoint_abs = f"/lakehouse/default/{checkpoint_location}"
    try:
        import shutil
        shutil.rmtree(checkpoint_abs)
        print(f"\U0001f5d1\ufe0f Cleared existing checkpoint: {checkpoint_location}")
    except FileNotFoundError:
        print(f"\u2139\ufe0f No existing checkpoint to clear at: {checkpoint_location}")

# Writer configuration (TARGET)
write_cfg = {
    "spark.cosmos.accountEndpoint": cosmos_target_endpoint,
    "spark.cosmos.accountKey": cosmos_target_masterkey,
    "spark.cosmos.applicationName": application_name,
    "spark.cosmos.database": cosmos_target_database_name,
    "spark.cosmos.container": cosmos_target_container_name,
    "spark.cosmos.write.strategy": "ItemOverwrite",
    "checkpointLocation": checkpoint_location,
}

# Read change feed
change_feed_df = (
    spark.readStream
    .format("cosmos.oltp.changeFeed")
    .options(**change_feed_cfg)
    .load()
)

# Preserve source _etag and _ts as audit fields
df_with_audit_fields = change_feed_df.withColumnRenamed("_rawbody", "_origin_rawBody")

# Shared read options for count queries (used here and in the streaming loop)
_source_read_opts = {
    "spark.cosmos.accountEndpoint": cosmos_source_endpoint,
    "spark.cosmos.accountKey": cosmos_source_masterkey,
    "spark.cosmos.database": cosmos_source_database_name,
    "spark.cosmos.container": cosmos_source_container_name,
    "spark.cosmos.read.inferSchema.enabled": "false",
    "spark.cosmos.preferredRegionsList": cosmos_region,
}
_target_read_opts = {
    "spark.cosmos.accountEndpoint": cosmos_target_endpoint,
    "spark.cosmos.accountKey": cosmos_target_masterkey,
    "spark.cosmos.database": cosmos_target_database_name,
    "spark.cosmos.container": cosmos_target_container_name,
    "spark.cosmos.read.inferSchema.enabled": "false",
    "spark.cosmos.preferredRegionsList": cosmos_region,
}

def count_container(opts):
    """Count documents in a Cosmos DB container."""
    return spark.read.format("cosmos.oltp").options(**opts).load().count()

# Initial counts
source_doc_count = count_container(_source_read_opts)
target_doc_count = count_container(_target_read_opts)
initial_gap = source_doc_count - target_doc_count

print(f"Change feed reader initialized for {cosmos_source_database_name}/{cosmos_source_container_name}")
print(f"Source document count: {source_doc_count:,}")
print(f"Target document count: {target_doc_count:,}")
print(f"Gap: {initial_gap:,}")
print(f"Checkpoint location: {checkpoint_location}")

## Step 6: Stream Migration with Count-Based Completion

The stream runs while periodic count queries on source and target containers determine completion. This approach is reliable regardless of:

- **Throttling** — slow batches don't affect the count comparison
- **Duplicate change feed events** — `numInputRows` is used only for display, never for completion
- **Active source writes** — stall detection prevents infinite loops; convergence retries close small gaps
- **Checkpoint resume** — counts reflect actual state, not stream progress counters

**Algorithm:**
1. Start the change feed stream (resumes from checkpoint if one exists)
2. Every `COUNT_CHECK_INTERVAL` seconds, query actual source and target document counts
3. If `target >= source` (within tolerance), stop the stream — migration complete
4. If the gap stops shrinking for `MAX_STALL_CHECKS` consecutive checks, stop and warn
5. Safety timeout prevents runaway execution

In [ ]:
import uuid
import time

# ---------------------------------------------------------------------------
# Tuning parameters
# ---------------------------------------------------------------------------
COUNT_CHECK_INTERVAL_SECONDS = 120   # how often to query source/target counts
PROGRESS_POLL_SECONDS = 10           # how often to print stream batch progress
MATCH_TOLERANCE = 100                # absolute doc-count tolerance for completion
MAX_STALL_CHECKS = 5                 # stop if gap doesn't shrink this many checks in a row
MAX_TOTAL_SECONDS = 86400            # 24-hour safety timeout

# ---------------------------------------------------------------------------
# If already fully migrated, skip streaming entirely
# ---------------------------------------------------------------------------
if initial_gap <= MATCH_TOLERANCE:
    print(f"\u2705 Target already has {target_doc_count:,} docs (source {source_doc_count:,}, "
          f"gap {initial_gap:,} within tolerance {MATCH_TOLERANCE:,}). Nothing to migrate.")
    final_source = source_doc_count
    final_target = target_doc_count
else:
    # ------------------------------------------------------------------
    # Start streaming write
    # ------------------------------------------------------------------
    run_id = str(uuid.uuid4())
    streaming_query = (
        df_with_audit_fields.writeStream
        .format("cosmos.oltp")
        .queryName(run_id)
        .options(**write_cfg)
        .outputMode("append")
        .start()
    )

    print(f"Streaming migration started: {cosmos_source_container_name} -> {cosmos_target_container_name}")
    print(f"Query ID: {run_id}")
    print(f"Initial gap: {initial_gap:,} docs  (source={source_doc_count:,}, target={target_doc_count:,})")
    print(f"Count check every {COUNT_CHECK_INTERVAL_SECONDS}s | "
          f"Stall limit: {MAX_STALL_CHECKS} checks | Tolerance: {MATCH_TOLERANCE:,} docs")
    print("=" * 80)

    prev_gap = initial_gap
    stall_count = 0
    start_time = time.time()
    last_count_check = start_time
    completion_status = "UNKNOWN"
    check_number = 0
    final_source = source_doc_count
    final_target = target_doc_count

    while streaming_query.isActive:
        time.sleep(PROGRESS_POLL_SECONDS)
        elapsed = time.time() - start_time

        # --- Print stream batch progress (display only, never used for completion) ---
        progress = streaming_query.lastProgress
        if progress:
            num_input_rows = progress.get("numInputRows", 0)
            if num_input_rows > 0:
                print(f"  [{elapsed:7.0f}s] Stream batch: {num_input_rows:,} rows processed")

        # --- Periodic count-based completion check ---
        if (time.time() - last_count_check) >= COUNT_CHECK_INTERVAL_SECONDS:
            last_count_check = time.time()
            check_number += 1

            try:
                current_source = count_container(_source_read_opts)
                current_target = count_container(_target_read_opts)
            except Exception as e:
                print(f"  [{elapsed:7.0f}s] Count check #{check_number} FAILED (will retry): {str(e)[:120]}")
                continue

            gap = current_source - current_target
            gap_delta = prev_gap - gap  # positive = gap is shrinking
            pct = (current_target / current_source * 100) if current_source > 0 else 100

            print(f"  [{elapsed:7.0f}s] Count check #{check_number}: "
                  f"source={current_source:,}  target={current_target:,}  "
                  f"gap={gap:,}  (\u0394={gap_delta:+,})  ({pct:.1f}%)")

            final_source = current_source
            final_target = current_target

            # -- Completion: target has caught up to source --
            if gap <= MATCH_TOLERANCE:
                completion_status = "COMPLETE"
                print(f"\n\u2705 Migration COMPLETE \u2014 gap {gap:,} within tolerance {MATCH_TOLERANCE:,}")
                streaming_query.stop()
                break

            # -- Stall detection: gap not shrinking --
            if gap >= prev_gap:
                stall_count += 1
                if stall_count >= MAX_STALL_CHECKS:
                    completion_status = "STALLED"
                    print(f"\n\u26a0\ufe0f Migration STALLED \u2014 gap has not shrunk for "
                          f"{MAX_STALL_CHECKS} consecutive checks ({gap:,} docs remaining). "
                          f"Source may be receiving writes faster than migration can copy.")
                    streaming_query.stop()
                    break
            else:
                stall_count = 0  # gap is shrinking, reset stall counter

            prev_gap = gap

        # --- Safety timeout ---
        if elapsed > MAX_TOTAL_SECONDS:
            completion_status = "TIMEOUT"
            # Get final counts before stopping
            try:
                final_source = count_container(_source_read_opts)
                final_target = count_container(_target_read_opts)
            except Exception:
                pass
            print(f"\n\u26a0\ufe0f Migration TIMEOUT after {MAX_TOTAL_SECONDS}s. "
                  f"source={final_source:,}, target={final_target:,}, "
                  f"gap={final_source - final_target:,}")
            streaming_query.stop()
            break

        # --- Stream stopped on its own (error, etc.) ---
        if not streaming_query.isActive:
            completion_status = "STREAM_STOPPED"
            break

    # Check for stream errors (InterruptedException is expected when we call .stop())
    query_exception = streaming_query.exception()
    if query_exception and "InterruptedException" not in str(query_exception):
        raise Exception(f"Streaming query failed: {query_exception}")

    # Final count verification (if we haven't just counted)
    if completion_status == "UNKNOWN" or completion_status == "STREAM_STOPPED":
        try:
            final_source = count_container(_source_read_opts)
            final_target = count_container(_target_read_opts)
        except Exception:
            pass

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
final_gap = final_source - final_target
print(f"\n{'=' * 80}")
print(f"Migration summary: {cosmos_source_container_name} -> {cosmos_target_container_name}")
print(f"  Source docs: {final_source:,}")
print(f"  Target docs: {final_target:,}")
print(f"  Gap:         {final_gap:,}")
if final_gap <= MATCH_TOLERANCE:
    exit_msg = (f"SUCCESS: {cosmos_source_container_name} -> {cosmos_target_container_name} "
                f"(source={final_source:,}, target={final_target:,}, gap={final_gap:,})")
    print(f"  Status:      \u2705 SUCCESS")
else:
    exit_msg = (f"WARNING: {cosmos_source_container_name} -> {cosmos_target_container_name} "
                f"(source={final_source:,}, target={final_target:,}, gap={final_gap:,}) \u2014 "
                f"run validation notebook to inspect missing docs")
    print(f"  Status:      \u26a0\ufe0f INCOMPLETE (run validation notebook)")
print(f"{'=' * 80}")

notebookutils.notebook.exit(exit_msg)